In [3]:
import torch
"""
T: Time / Context Length / Block Size / Sequence Length
    Max size of context model will support to generate a new token.
    Its a "max" size. 
    If T = 128, our training data samples has input context size from 1 to 128.
    i.e.: T = 8
    Context: [60] | Target: 47
    Context: [60, 47] | Target: 1
    Context: [60, 47, 1] | Target: 44
    Context: [60, 47, 1, 44] | Target: 43
    Context: [60, 47, 1, 44, 43] | Target: 43
    Context: [60, 47, 1, 44, 43, 43] | Target: 54
    Context: [60, 47, 1, 44, 43, 43, 54] | Target: 1
    Context: [60, 47, 1, 44, 43, 43, 54, 1] | Target: 58

B: Batch Size 
    Since transformer can be trained in parallel,
    Instead of doing calculation/training for a single text sequence, 
    we can do for multilple text sequences.
    We set value of B based on the our GPU vRAM. (First set the context window according to 
    our model requirement and lets say it is using 1.3 GB of vRAM out of 12.
    Then for B = 2, 4, 8, 16 vRAM usage: 2.6, 5.2, 10.4, 20.8   
    We set B = 8

C: Channel
    Embeddings of a token
    C represents the embeding size/length
    

"""

B, T, C = 4,8,2  # batch, time, channel

# Returns a tensor filled with random numbers from a normal distribution 
#  with mean 0 and variance 1 (also called the standard normal distribution).
x = torch.randn(B, T, C)
print(x.shape)
x

torch.Size([4, 8, 2])


tensor([[[-0.6900, -1.2998],
         [-0.1374,  0.2000],
         [-1.7794,  0.3376],
         [ 1.6148,  0.1407],
         [ 1.4319,  0.4122],
         [-0.1145,  1.5174],
         [-2.5709, -0.3749],
         [-0.9195, -1.3828]],

        [[-0.9557, -1.9190],
         [ 0.1496, -0.3711],
         [-0.2047,  0.5778],
         [-0.8253, -0.4555],
         [ 0.8528, -1.0397],
         [-1.7985,  1.2713],
         [-0.8202, -0.5449],
         [ 0.1869,  1.2814]],

        [[ 1.5251, -0.5335],
         [-1.2650,  0.9201],
         [ 0.3560,  1.4437],
         [-0.7348,  1.5779],
         [-1.0174,  1.2035],
         [ 0.1920,  0.9317],
         [ 0.1758, -0.7116],
         [ 0.9197, -2.3226]],

        [[-0.4409,  1.9318],
         [-2.6442, -1.0555],
         [-0.4754,  1.4395],
         [-0.5158,  0.7566],
         [-0.2175, -0.4108],
         [ 1.0081, -0.3446],
         [ 0.6985,  0.1113],
         [-0.3591,  0.9016]]])

In [7]:
x[:, -1, :]

tensor([[-0.9195, -1.3828],
        [ 0.1869,  1.2814],
        [ 0.9197, -2.3226],
        [-0.3591,  0.9016]])

In [9]:

loss = torch.nn.CrossEntropyLoss()
input = torch.randn(3, 5, requires_grad=True)
target = torch.empty(3, dtype=torch.long).random_(5)
output = loss(input, target)
output


tensor(1.3702, grad_fn=<NllLossBackward0>)

In [10]:
input

tensor([[ 0.1664,  2.4532,  0.7853, -1.8967, -1.0262],
        [ 0.2687,  1.0319, -0.0598, -0.7823,  2.4150],
        [-2.3041,  0.1343, -0.4177,  0.0797,  0.1455]], requires_grad=True)

In [28]:
torch.randn(3, 500000).mean(dim=1)

tensor([0.0009, 0.0034, 0.0017])

In [31]:
table = torch.nn.Embedding(10, 12)
table(torch.tensor(2))

tensor([ 0.3542, -1.9401, -0.6175, -0.0686, -0.2019,  1.1829,  0.7719, -1.2521,
        -0.1171,  0.6077, -0.3487, -1.3761], grad_fn=<EmbeddingBackward0>)

In [38]:
a = torch.ones(3,3)
b = torch.randint(0, 10, (3,2)).float()
c = a @ b

print(a, end= "\n")
print(b, end= "\n")
print(c, end= "\n")

tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
tensor([[7., 8.],
        [4., 4.],
        [5., 3.]])
tensor([[16., 15.],
        [16., 15.],
        [16., 15.]])


In [41]:
a = torch.tril(torch.ones(3,3))
b = torch.randint(0, 10, (3,2)).float()
c = a @ b

print(a, end= "\n")
print(b, end= "\n")
print(c, end= "\n")

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
tensor([[5., 8.],
        [9., 9.],
        [3., 4.]])
tensor([[ 5.,  8.],
        [14., 17.],
        [17., 21.]])


In [42]:
"""
Above in matrix c, we are doing  sum of values in matrix B verticaly/column wise
first row in matrix C: 5, 8
then in 2nd row C: 14, 17 > (5+9 and 8+9)
then in 3rd row C: 17, 21 > (5+9+3 and 8+9+4)
"""

'\nAbove in matrix c, we are doing  sum of values in matrix B verticaly/column wise\nfirst row in matrix C: 5, 8\nthen in 2nd row C: 14, 17 > (5+9 and 8+9)\nthen in 3rd row C: 17, 21 > (5+9+3 and 8+94)\n'

In [47]:
# Instead of sum, we can do average also by normalizing matrix a
a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, dim=1, keepdim=True)
b = torch.randint(0, 10, (3,2)).float()
c = a @ b

print(a, end= "\n")
print(b, end= "\n")
print(c, end= "\n") 

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[6., 1.],
        [7., 4.],
        [1., 3.]])
tensor([[6.0000, 1.0000],
        [6.5000, 2.5000],
        [4.6667, 2.6667]])


## Version 2: using tril

In [50]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
print(wei)

# here wei.shape: (T, T) and x.shape: (B, T, C)
# so pytorch will see that these imensions are not same, so it wil create a batch dim. now wei.shape: (B, T, T)
# final xbow.shape will b e (B, T, C)
# so now we get the average values for matrix x columns wise 
xbow = wei @ x
xbow

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


tensor([[[-0.6900, -1.2998],
         [-0.4137, -0.5499],
         [-0.8689, -0.2541],
         [-0.2480, -0.1554],
         [ 0.0880, -0.0419],
         [ 0.0542,  0.2180],
         [-0.3208,  0.1333],
         [-0.3956, -0.0562]],

        [[-0.9557, -1.9190],
         [-0.4031, -1.1451],
         [-0.3370, -0.5708],
         [-0.4590, -0.5419],
         [-0.1967, -0.6415],
         [-0.4636, -0.3227],
         [-0.5146, -0.3544],
         [-0.4269, -0.1500]],

        [[ 1.5251, -0.5335],
         [ 0.1301,  0.1933],
         [ 0.2054,  0.6101],
         [-0.0297,  0.8521],
         [-0.2272,  0.9223],
         [-0.1574,  0.9239],
         [-0.1098,  0.6903],
         [ 0.0189,  0.3137]],

        [[-0.4409,  1.9318],
         [-1.5426,  0.4381],
         [-1.1868,  0.7719],
         [-1.0191,  0.7681],
         [-0.8587,  0.5323],
         [-0.5476,  0.3862],
         [-0.3696,  0.3469],
         [-0.3683,  0.4162]]])

In [49]:
x[0], xbow[0]

(tensor([[-0.6900, -1.2998],
         [-0.1374,  0.2000],
         [-1.7794,  0.3376],
         [ 1.6148,  0.1407],
         [ 1.4319,  0.4122],
         [-0.1145,  1.5174],
         [-2.5709, -0.3749],
         [-0.9195, -1.3828]]),
 tensor([[-0.6900, -1.2998],
         [-0.4137, -0.5499],
         [-0.8689, -0.2541],
         [-0.2480, -0.1554],
         [ 0.0880, -0.0419],
         [ 0.0542,  0.2180],
         [-0.3208,  0.1333],
         [-0.3956, -0.0562]]))

## Version 3: using softmax

In [51]:
tril = torch.tril(torch.ones(T, T))
print(tril)
wei = torch.zeros(T, T)
print(wei)
wei = wei.masked_fill(tril == 0, float("-inf"))
print(wei)
wei = torch.nn.functional.softmax(wei, dim=-1)
print(wei)

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])
tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0.,

In [52]:
"""
Above approach is recommended to use.
Because in this we are starting with wei = zeros, then we are increasing it to number 
which tells us how much tokens we want to average out.
Then masking tell us no to use the future tokens.
Then we apply softmax to normalize the values.

Here we set wei = zeros
But in reality (q@k.t) tokens look at each other and then increase its values if they find more related to each other 
"""

'\nAbove approach is recommended to use.\nBecause in this we are starting with wei = zeros, then we are increasing it to number \nwhich tells us how much tokens we want to average out.\nThen masking tell us no to use the future tokens.\nThen we apply softmax to normalize the values.\n\nHere we set wei = zeros\nBut in reality tokens look at each other and then increase its values if they find more related to each other \n'

## Implemention of self attention (single head)

In [67]:
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16 # usualy this is decide by embedding size divide by no of heads in multi-headed attention (paper: 512/8=64)
key = torch.nn.Linear(C, head_size, bias=False)
query = torch.nn.Linear(C, head_size, bias=False)

k = key(x) # (B, T, 16)
q = query(x) # (B, T, 16)
# above all the tokens at any position of (B,T) has produced a key and a query for size 16

# now we will do the communication between k and q for each token
# k.T will result in trasnpose of btach as well, so we need to specify the dimensions in which we need transpose.
wei = q @ k.transpose(-2, -1) # (B, T, 16)@(B, 16, T) ====> (B, T, T)
print(wei[0]) # printing first batch

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
print(wei[0])
wei = torch.nn.functional.softmax(wei, dim=-1)
print(wei[0])


tensor([[-1.6087, -1.9737,  2.0678, -0.0835,  0.5123,  0.6910,  0.1251, -2.7924],
        [ 2.4575, -0.4961, -2.0848,  1.3111, -0.1809, -0.3335,  0.3758,  2.2745],
        [ 1.3380,  1.3900, -0.1986,  0.0378, -1.3682, -1.2785,  1.1836,  1.2201],
        [ 1.3833, -1.4084,  0.5704,  0.0693,  1.4472, -0.9067, -0.9965,  0.2856],
        [-2.1079,  2.1485,  0.3416,  0.4166, -0.3529,  1.1118, -0.1464, -0.0029],
        [-0.1130, -0.4891,  0.2130, -0.3483, -0.2733,  1.6130, -0.1548,  0.4063],
        [-0.1757,  1.6756, -0.6309,  1.3031, -1.0570, -0.6621, -0.4889,  0.0930],
        [ 1.5229, -0.3710, -0.9170, -1.3223,  0.4644,  2.0606, -0.1400,  0.3225]],
       grad_fn=<SelectBackward0>)
tensor([[-1.6087,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 2.4575, -0.4961,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 1.3380,  1.3900, -0.1986,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 1.3833, -1.4084,  0.5704,  0.0693,    -inf,    -inf, 

In [72]:
"""
Adding Value
"""
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16 # usualy this is decide by embedding size divide by no of heads in multi-headed attention (paper: 512/8=64)
key = torch.nn.Linear(C, head_size, bias=False)
query = torch.nn.Linear(C, head_size, bias=False)
value = torch.nn.Linear(C, head_size, bias=False)

k = key(x) # (B, T, 16)
q = query(x) # (B, T, 16)
v = value(x)
# above all the tokens at any position of (B,T) has produced a key and a query for size 16

# now we will do the communication between k and q for each token
# k.T will result in trasnpose of btach as well, so we need to specify the dimensions in which we need transpose.
wei = q @ k.transpose(-2, -1) # (B, T, 16)@(B, 16, T) ====> (B, T, T)
print(wei[0]) # printing first batch

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
print(wei[0])
wei = torch.nn.functional.softmax(wei, dim=-1)
print(wei[0])

output = wei @ v
print(output.shape)
output[0]

tensor([[ 0.8756,  0.8391,  0.8844, -1.3296,  1.4566, -0.5038,  1.0197, -0.7838],
        [-0.7522,  0.8905,  1.3667,  1.2740,  0.8396, -2.4906,  0.0510, -0.5351],
        [ 0.0793, -0.5671, -1.1780, -0.1969, -1.1745, -0.0366, -0.1634,  2.8698],
        [ 0.9461,  0.6561, -0.6807, -0.6550,  0.2063,  0.2852, -1.7964, -0.9213],
        [ 2.0832,  0.1229, -0.8644, -1.2835, -0.7605,  2.0028, -0.0928, -0.1316],
        [ 1.7518,  0.4554, -0.8044,  1.0591, -1.5058,  0.0416, -0.4275,  0.5311],
        [ 1.2980, -0.0075, -1.8427, -1.2215,  0.9975,  3.0242, -0.0593,  0.9756],
        [ 0.9957,  0.8582, -0.4659,  0.7155,  0.9588,  0.2674,  0.8344, -0.4903]],
       grad_fn=<SelectBackward0>)
tensor([[ 0.8756,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.7522,  0.8905,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.0793, -0.5671, -1.1780,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.9461,  0.6561, -0.6807, -0.6550,    -inf,    -inf, 

tensor([[-0.1447,  0.7200, -0.7210,  0.9177, -0.1384,  0.7392,  1.3311,  0.3671,
         -0.6615, -0.7190, -0.3611, -0.6083, -0.0686, -0.5612, -0.7435, -0.3285],
        [-0.8763,  0.0432, -0.6275,  0.3745, -0.5233, -0.4569,  0.3938, -0.2473,
         -0.9427, -0.4426,  0.8068, -0.7569,  0.2490,  0.3827, -0.2314, -0.6438],
        [-0.2323,  0.3809, -0.6664,  0.5073, -0.3518,  0.2704,  0.6912,  0.1249,
         -0.6566, -0.4464,  0.1112, -0.6407,  0.0437, -0.1024, -0.3003, -0.4439],
        [-0.3750,  0.3119, -0.7079,  0.6097, -0.3468,  0.0756,  0.5751, -0.0128,
         -0.6079, -0.4444,  0.2349, -0.5490,  0.1316, -0.1148, -0.2944, -0.4776],
        [-0.1752,  0.5652, -0.6815,  0.7375, -0.2078,  0.4856,  1.0113,  0.2441,
         -0.6202, -0.5638, -0.1679, -0.5391,  0.0301, -0.3908, -0.5395, -0.3397],
        [-0.3590,  0.3095, -0.7510,  0.8009, -0.1434,  0.0388,  0.5618, -0.0475,
         -0.3922, -0.3821,  0.0741, -0.3320,  0.1735, -0.4236, -0.3142, -0.5008],
        [-0.8282, -0.2

### Why we divide wei by sqrt(head_size)

In [78]:
k = torch.randn(T, head_size)
q = torch.randn(T, head_size)

wei = q @ k.T

In [79]:
k.var(), q.var(), wei.var()

(tensor(0.8098), tensor(1.0342), tensor(16.3502))

In [80]:
wei = wei * head_size**-0.5
wei.var()

tensor(1.0219)

In [82]:
"""
Why we need less variance in wei?
Becuase later we apply softmax on wei. 
So more vairance the values in wei has, more softmax will converge into one-hot encoding vectors 
towards the max value of wei.
Sotmax will be way too sharp in direction of max value.

we do not need this. Because in this case every node will be aggregating the information form a sinlge node.
Which we do not want, specially at the initial stage of training.
"""

'\nWhy we need less variance in wei?\nBecuase later we apply softmax on wei. \nSo more vairance the values in wei has, more softmax will converge into one-hot encoding vectors \ntowards the max value of wei.\n'

In [86]:
print(torch.softmax(torch.tensor([0.2, -0.2]), dim=-1))
print(torch.softmax(torch.tensor([0.4, -0.4]), dim=-1))
print(torch.softmax(torch.tensor([0.8, -0.8]), dim=-1))

tensor([0.5987, 0.4013])
tensor([0.6900, 0.3100])
tensor([0.8320, 0.1680])


In [89]:
print(torch.softmax(torch.tensor([0.2, 0.1, -0.3, -0.2, 0.8]), dim=-1))
print(torch.softmax(torch.tensor([0.2, 0.1, -0.3, -0.2, 0.8])*5, dim=-1))

tensor([0.1998, 0.1808, 0.1212, 0.1340, 0.3641])
tensor([0.0456, 0.0277, 0.0037, 0.0062, 0.9168])
